<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Console.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# [Colab 貼心提醒] 如果尚未安裝所需套件，請先在 Colab 執行以下指令安裝：
# !pip install -q yfinance openpyxl pandas
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from google.colab import files

warnings.filterwarnings("ignore")

# ==============================================================================
# 0. 全局交易設定與標的池
# ==============================================================================
TOTAL_PORTFOLIO_CAPITAL = (
    10000000  # 擬投入組合總資金 (例如：NTD 10,000,000)
)

taiwan_tech_tickers = [
    "2330.TW",
    "2317.TW",
    "2454.TW",
    "2382.TW",
    "3231.TW",
    "2308.TW",
    "2303.TW",
    "3711.TW",
    "2357.TW",
    "2395.TW",
    "3034.TW",
    "2379.TW",
    "3035.TW",
    "3443.TW",
    "6669.TW",
    "6415.TW",
    "3661.TW",
    "3037.TW",
    "8046.TW",
    "3189.TW",
    "2345.TW",
    "6285.TW",
    "5388.TW",
    "3044.TW",
    "2368.TW",
    "6213.TW",
    "6274.TWO",
    "2383.TW",
    "3017.TW",
    "3324.TWO",
    "3019.TW",
    "6176.TW",
    "3406.TW",
    "2327.TW",
    "2492.TW",
    "3026.TW",
    "2360.TW",
    "6239.TW",
    "2449.TW",
    "6257.TW",
]

taiwan_traditional_tickers = [
    "1101.TW",
    "1102.TW",
    "1216.TW",
    "1301.TW",
    "1303.TW",
    "1326.TW",
    "2002.TW",
    "2105.TW",
    "2207.TW",
    "2603.TW",
    "2609.TW",
    "2615.TW",
    "2618.TW",
    "2610.TW",
    "9910.TW",
    "9904.TW",
    "1402.TW",
    "2912.TW",
    "2201.TW",
    "2204.TW",
    "1314.TW",
    "1717.TW",
    "1722.TW",
    "1710.TW",
    "1304.TW",
    "1308.TW",
    "2103.TW",
    "2106.TW",
    "8454.TW",
    "9921.TW",
    "9914.TW",
    "9924.TW",
    "9930.TW",
    "8936.TWO",
    "8464.TW",
    "1504.TW",
    "1513.TW",
    "1514.TW",
    "1519.TW",
    "1609.TW",
]

us_tech_tickers = [
    "NVDA",
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "META",
    "TSLA",
    "AVGO",
    "AMD",
    "QCOM",
    "TXN",
    "ADI",
    "MU",
    "AMAT",
    "LRCX",
    "KLAC",
    "INTC",
    "ARM",
    "TSM",
    "ASML",
    "ORCL",
    "PANW",
    "CRWD",
    "SNOW",
    "PLTR",
    "DDOG",
    "NET",
    "ZS",
    "FTNT",
    "SMCI",
    "DELL",
    "HPE",
    "ANET",
    "CSCO",
    "IBM",
    "NOW",
    "CRM",
    "ADBE",
    "INTU",
    "ACN",
]

us_traditional_tickers = [
    "JNJ",
    "PFE",
    "UNH",
    "ABBV",
    "MRK",
    "LLY",
    "PG",
    "KO",
    "PEP",
    "COST",
    "WMT",
    "TGT",
    "HD",
    "LOW",
    "MCD",
    "SBUX",
    "NKE",
    "CAT",
    "DE",
    "GE",
    "HON",
    "MMM",
    "UPS",
    "FDX",
    "BA",
    "LMT",
    "RTX",
    "XOM",
    "CVX",
    "COP",
    "SLB",
    "EOG",
    "JPM",
    "BAC",
    "WFC",
    "C",
    "GS",
    "MS",
    "BLK",
    "V",
]

all_tickers = (
    taiwan_tech_tickers
    + taiwan_traditional_tickers
    + us_tech_tickers
    + us_traditional_tickers
)

ticker_to_sector = {}
for t in taiwan_tech_tickers:
    ticker_to_sector[t] = "台股電子/半導體/供應鏈"
for t in taiwan_traditional_tickers:
    ticker_to_sector[t] = "台股傳統/金融/航運/產化"
for t in us_tech_tickers:
    ticker_to_sector[t] = "美股科技/半導體/雲端軟體"
for t in us_traditional_tickers:
    ticker_to_sector[t] = "美股消費/醫療/工業/金融/能源"

ticker_to_name = {
    "2330.TW": "台積電",
    "2317.TW": "鴻海",
    "2454.TW": "聯發科",
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "2308.TW": "台達電",
    "2303.TW": "聯電",
    "3711.TW": "日月光投控",
    "2357.TW": "華碩",
    "2395.TW": "研華",
    "3034.TW": "聯詠",
    "2379.TW": "瑞昱",
    "3035.TW": "智原",
    "3443.TW": "創意",
    "6669.TW": "緯穎",
    "6415.TW": "矽力-KY",
    "3661.TW": "世芯-KY",
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "2345.TW": "智邦",
    "6285.TW": "啟碁",
    "5388.TW": "中磊",
    "3044.TW": "健鼎",
    "2368.TW": "金像電",
    "6213.TW": "聯茂",
    "6274.TWO": "台燿",
    "2383.TW": "台光電",
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3019.TW": "亞光",
    "6176.TW": "瑞儀",
    "3406.TW": "玉晶光",
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "3026.TW": "禾伸堂",
    "2360.TW": "致茂",
    "6239.TW": "力成",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "1101.TW": "台泥",
    "1102.TW": "亞泥",
    "1216.TW": "統一",
    "1301.TW": "台塑",
    "1303.TW": "南亞",
    "1326.TW": "台化",
    "2002.TW": "中鋼",
    "2105.TW": "正新",
    "2207.TW": "和泰車",
    "2603.TW": "長榮",
    "2609.TW": "陽明",
    "2615.TW": "萬海",
    "2618.TW": "長榮航",
    "2610.TW": "華航",
    "9910.TW": "豐泰",
    "9904.TW": "寶成",
    "1402.TW": "遠東新",
    "2912.TW": "統一超",
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "1314.TW": "中石化",
    "1717.TW": "長興",
    "1722.TW": "台肥",
    "1710.TW": "東聯",
    "1304.TW": "台聚",
    "1308.TW": "亞聚",
    "2103.TW": "台橡",
    "2106.TW": "建大",
    "8454.TW": "富邦媒",
    "9921.TW": "巨大",
    "9914.TW": "美利達",
    "9924.TW": "福興",
    "9930.TW": "中聯資源",
    "8936.TWO": "國統",
    "8464.TW": "億豐",
    "1504.TW": "東元",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1519.TW": "華城",
    "1609.TW": "太電",
    "NVDA": "NVIDIA",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "GOOGL": "Alphabet",
    "AMZN": "Amazon",
    "META": "Meta Platforms",
    "TSLA": "Tesla",
    "AVGO": "Broadcom",
    "AMD": "AMD",
    "QCOM": "Qualcomm",
    "TXN": "Texas Instruments",
    "ADI": "Analog Devices",
    "MU": "Micron",
    "AMAT": "Applied Materials",
    "LRCX": "Lam Research",
    "KLAC": "KLA Corp",
    "INTC": "Intel",
    "ARM": "Arm Holdings",
    "TSM": "TSMC ADR",
    "ASML": "ASML",
    "ORCL": "Oracle",
    "PANW": "Palo Alto Networks",
    "CRWD": "CrowdStrike",
    "SNOW": "Snowflake",
    "PLTR": "Palantir",
    "DDOG": "Datadog",
    "NET": "Cloudflare",
    "ZS": "Zscaler",
    "FTNT": "Fortinet",
    "SMCI": "Super Micro Computer",
    "DELL": "Dell Technologies",
    "HPE": "Hewlett Packard Enterprise",
    "ANET": "Arista Networks",
    "CSCO": "Cisco",
    "IBM": "IBM",
    "NOW": "ServiceNow",
    "CRM": "Salesforce",
    "ADBE": "Adobe",
    "INTU": "Intuit",
    "ACN": "Accenture",
    "JNJ": "Johnson & Johnson",
    "PFE": "Pfizer",
    "UNH": "UnitedHealth Group",
    "ABBV": "AbbVie",
    "MRK": "Merck & Co.",
    "LLY": "Eli Lilly",
    "PG": "Procter & Gamble",
    "KO": "Coca-Cola",
    "PEP": "PepsiCo",
    "COST": "Costco",
    "WMT": "Walmart",
    "TGT": "Target",
    "HD": "Home Depot",
    "LOW": "Lowe's",
    "MCD": "McDonald's",
    "SBUX": "Starbucks",
    "NKE": "Nike",
    "CAT": "Caterpillar",
    "DE": "John Deere",
    "GE": "GE Aerospace",
    "HON": "Honeywell",
    "MMM": "3M",
    "UPS": "UPS",
    "FDX": "FedEx",
    "BA": "Boeing",
    "LMT": "Lockheed Martin",
    "RTX": "RTX Corp",
    "XOM": "ExxonMobil",
    "CVX": "Chevron",
    "COP": "ConocoPhillips",
    "SLB": "SLB",
    "EOG": "EOG Resources",
    "JPM": "JPMorgan Chase",
    "BAC": "Bank of America",
    "WFC": "Wells Fargo",
    "C": "Citigroup",
    "GS": "Goldman Sachs",
    "MS": "Morgan Stanley",
    "BLK": "BlackRock",
    "V": "Visa",
}


# ==============================================================================
# 1. 模擬歷史預測與風控數據 (保持原本 21 個欄位)
# ==============================================================================
def simulate_base_prediction_pipeline(tickers):
    np.random.seed(42)
    n = len(tickers)

    p_pred_raw = np.random.uniform(0.40, 0.85, n)
    win_rate = p_pred_raw * np.random.uniform(0.85, 0.98, n)
    avg_win = np.random.uniform(0.03, 0.08, n)
    avg_loss = np.random.uniform(0.015, 0.035, n)

    eval_score = (win_rate * avg_win) / (avg_loss + 1e-5) * 10
    prob_threshold = 0.55

    is_valid_signal = np.where(p_pred_raw >= prob_threshold, "有效交易", "濾除觀望")
    base_weight = np.where(is_valid_signal == "有效交易", p_pred_raw * 0.2, 0.0)
    sector_cap = 0.25

    records = []
    for i, t in enumerate(tickers):
        sec = ticker_to_sector.get(t, "未知產業")
        name = ticker_to_name.get(t, t)

        rec = {
            "股票代號": t,
            "股票名稱": name,
            "產業分類": sec,
            "p_pred_raw": round(p_pred_raw[i], 4),
            "模型判定看漲機率": f"{p_pred_raw[i]*100:.2f}%",
            "機率閾值(濾除噪訊)": f"{prob_threshold*100:.1f}%",
            "勝率(Win Rate)": round(win_rate[i], 4),
            "平均獲利(Avg Win)": round(avg_win[i], 4),
            "平均損失(Avg Loss)": round(avg_loss[i], 4),
            "評估分數": round(eval_score[i], 2),
            "專業優化訊號有效性": is_valid_signal[i],
            "原始建議部位": f"{base_weight[i]*100:.2f}%",
            "單一產業上限": f"{sector_cap*100:.0f}%",
            "勝率_num": win_rate[i],
            "avg_win": avg_win[i],
            "avg_loss": avg_loss[i],
            "eval_score": eval_score[i],
            "base_weight": base_weight[i],
        }
        records.append(rec)

    df = pd.DataFrame(records)

    df["sector_total_weight"] = df.groupby("產業分類")["base_weight"].transform(
        "sum"
    )
    df["sector_scale"] = np.where(
        df["sector_total_weight"] > sector_cap,
        sector_cap / (df["sector_total_weight"] + 1e-9),
        1.0,
    )
    df["rebalanced_weight"] = df["base_weight"] * df["sector_scale"]

    df["風控頂格再平衡建議部位比率"] = (df["rebalanced_weight"] * 100).map(
        "{:.2f}%".format
    )
    df["風控頂格再平衡建議部位_num"] = df["rebalanced_weight"] * 100
    df["風控狀態描述"] = np.where(
        df["sector_scale"] < 1.0,
        "觸及產業上限(已等比縮減)",
        "風控正常(未觸及上限)",
    )

    df["5日3%回報率模型勝率"] = df["勝率(Win Rate)"].map("{:.2f}%".format)
    df["5日平均獲利幅度"] = df["平均獲利(Avg Win)"].map("{:.2f}%".format)
    df["5日平均虧損幅度"] = df["平均損失(Avg Loss)"].map("{:.2f}%".format)
    df["綜合期望值評估分數"] = df["評估分數"]

    cols_order = [
        "股票代號",
        "股票名稱",
        "產業分類",
        "模型判定看漲機率",
        "機率閾值(濾除噪訊)",
        "專業優化訊號有效性",
        "5日3%回報率模型勝率",
        "5日平均獲利幅度",
        "5日平均虧損幅度",
        "綜合期望值評估分數",
        "原始建議部位",
        "單一產業上限",
        "風控頂格再平衡建議部位比率",
        "風控狀態描述",
        "p_pred_raw",
        "勝率_num",
        "avg_win",
        "avg_loss",
        "eval_score",
        "base_weight",
        "風控頂格再平衡建議部位_num",
    ]
    return df[cols_order]


# ==============================================================================
# 2. 交易員與 PM 級別「可執行交易報表」優化模組 (向量化批次下載)
# ==============================================================================
def enrich_trader_execution_sheet(df_res, total_portfolio_capital=10000000):
    df_trade = df_res.copy()

    print(
        "🔍 正在一次性向量化拉取 160 隻標的最新行情與波動度數據 (yfinance Batch Download)..."
    )

    tickers_list = df_trade["股票代號"].tolist()

    try:
        data = yf.download(
            tickers_list, period="1mo", progress=False, group_by="ticker"
        )
    except Exception as e:
        print(f"⚠️ 網路下載數據異常，使用預設市價填補: {e}")
        data = None

    ref_prices = []
    atrs = []
    advs = []

    for ticker in tickers_list:
        try:
            if data is not None and ticker in data:
                df_t = data[ticker].dropna()
                if not df_t.empty and len(df_t) >= 5:
                    latest_close = float(df_t["Close"].iloc[-1])
                    tr = np.maximum(
                        df_t["High"] - df_t["Low"],
                        np.maximum(
                            abs(df_t["High"] - df_t["Close"].shift(1)),
                            abs(df_t["Low"] - df_t["Close"].shift(1)),
                        ),
                    )
                    atr = float(tr.rolling(min(14, len(tr))).mean().iloc[-1])
                    adv = float(
                        (df_t["Close"] * df_t["Volume"]).tail(20).mean()
                    )
                else:
                    latest_close, atr, adv = (
                        (500.0 if ".TW" in ticker else 150.0),
                        10.0,
                        500000000.0,
                    )
            else:
                latest_close, atr, adv = (
                    (500.0 if ".TW" in ticker else 150.0),
                    10.0,
                    500000000.0,
                )
        except Exception:
            latest_close, atr, adv = (
                (500.0 if ".TW" in ticker else 150.0),
                10.0,
                500000000.0,
            )

        ref_prices.append(latest_close)
        atrs.append(atr)
        advs.append(adv)

    df_trade["最新參考價"] = ref_prices
    df_trade["ATR_14"] = atrs
    df_trade["20日均成交額"] = advs

    rebalance_pct = df_trade["風控頂格再平衡建議部位_num"] / 100.0
    conditions = [
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct >= 0.05),
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct > 0.0),
    ]
    choices = ["STRONG_BUY", "BUY_NEW"]
    df_trade["建議交易動作"] = np.select(conditions, choices, default="NO_ACTION")
    df_trade["目標分配金額"] = total_portfolio_capital * rebalance_pct

    def calc_shares(row):
        price = row["最新參考價"]
        capital = row["目標分配金額"]
        ticker = str(row["股票代號"])

        if pd.isna(price) or price <= 0 or capital <= 0:
            return "0 股"

        raw_shares = capital / price
        if ".TW" in ticker or ".TWO" in ticker:
            lots = int(raw_shares // 1000)
            odd_shares = int(raw_shares % 1000)
            if lots > 0:
                return (
                    f"{lots} 張 ({odd_shares} 股)"
                    if odd_shares > 0
                    else f"{lots} 張"
                )
            else:
                return f"{odd_shares} 股"
        else:
            return f"{int(raw_shares)} 股"

    df_trade["預估下單數量"] = df_trade.apply(calc_shares, axis=1)
    df_trade["建議停損價(SL)"] = (
        df_trade["最新參考價"] - (1.5 * df_trade["ATR_14"])
    ).round(2)
    df_trade["建議停利價(TP)"] = (
        df_trade["最新參考價"] * (1 + df_trade["avg_win"])
    ).round(2)

    df_trade["日均成交額占比(%)"] = (
        (df_trade["目標分配金額"] / (df_trade["20日均成交額"] + 1e-9)) * 100
    ).round(2)
    df_trade["流動性風險評估"] = np.where(
        df_trade["日均成交額占比(%)"] > 5.0,
        "高衝擊(建議分批)",
        "良好",
    )

    df_trade["目標分配金額(NTD/USD)"] = df_trade["目標分配金額"].apply(
        lambda x: f"${x:,.0f}"
    )
    df_trade["最新參考價_fmt"] = df_trade["最新參考價"].round(2)

    df_trade.sort_values(
        by=["風控頂格再平衡建議部位_num", "p_pred_raw"],
        ascending=[False, False],
        inplace=True,
    )

    # 輸出機構級多頁面 Excel 檔 (Workbook)
    filename = "Portfolio_Execution_Order_Sheet.xlsx"
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        execution_mask = df_trade["建議交易動作"].isin(
            ["STRONG_BUY", "BUY_NEW"]
        )
        execution_cols = [
            "股票代號",
            "股票名稱",
            "產業分類",
            "建議交易動作",
            "最新參考價_fmt",
            "目標分配金額(NTD/USD)",
            "預估下單數量",
            "建議停損價(SL)",
            "建議停利價(TP)",
            "風控頂格再平衡建議部位比率",
            "流動性風險評估",
        ]
        execution_rename = {"最新參考價_fmt": "最新參考價"}

        df_exec = df_trade[execution_mask][execution_cols].rename(
            columns=execution_rename
        )

        # Tab 1: Top 10 交易執行清單 (Top 10 Orders)
        df_top10 = df_exec.head(10)
        df_top10.to_excel(writer, sheet_name="Top10_Orders", index=False)

        # Tab 2: 完整交易員執行頁 (Trader Execution Sheet)
        df_exec.to_excel(
            writer, sheet_name="Trader_Execution_Orders", index=False
        )

        # Tab 3: 基金經理人完整風控頁 (PM Risk & Analytical Sheet)
        df_trade.to_excel(writer, sheet_name="PM_Full_Analysis", index=False)

    print(f"✅ 已成功整合專業交易欄位！可執行報表已匯出至：{filename}\n")
    return df_trade, df_top10, filename


# ==============================================================================
# 3. 主執行流程 (Main Pipeline)
# ==============================================================================
if __name__ == "__main__":
    print("🚀 啟動量化預測與交易報表生成系統...")

    # Step 1: 執行既有模型預測 Pipeline
    df_base_result = simulate_base_prediction_pipeline(all_tickers)

    # Step 2: 注入專業交易員/PM 優化模組
    df_final_trade, df_top10, excel_filename = enrich_trader_execution_sheet(
        df_base_result, total_portfolio_capital=TOTAL_PORTFOLIO_CAPITAL
    )

    # Step 3: Console 展示 Top 10
    print("=" * 100)
    print("📋【交易員開盤可執行下單清單 (Top 10)】")
    print("=" * 100)
    print(df_top10.to_string(index=False))
    print("=" * 100)

    # Step 4: 觸發 Google Colab 網頁直接下載 Excel 附件
    print(f"📥 修正完成！正在啟動 Google Colab 檔案下載：{excel_filename} ...")
    files.download(excel_filename)


🚀 啟動量化預測與交易報表生成系統...
🔍 正在一次性向量化拉取 160 隻標的最新行情與波動度數據 (yfinance Batch Download)...
✅ 已成功整合專業交易欄位！可執行報表已匯出至：Portfolio_Execution_Order_Sheet.xlsx

📋【交易員開盤可執行下單清單 (Top 10)】
   股票代號         股票名稱             產業分類  建議交易動作  最新參考價 目標分配金額(NTD/USD)      預估下單數量  建議停損價(SL)  建議停利價(TP) 風控頂格再平衡建議部位比率 流動性風險評估
2379.TW           瑞昱     台股電子/半導體/供應鏈 BUY_NEW 713.00        $129,747       181 股     670.89     761.41         1.30%      良好
    WFC  Wells Fargo 美股消費/醫療/工業/金融/能源 BUY_NEW  90.29        $129,683      1436 股      87.61      95.38         1.30%      良好
2492.TW          華新科     台股電子/半導體/供應鏈 BUY_NEW 305.50        $129,449       423 股     269.98     315.74         1.29%      良好
     GE GE Aerospace 美股消費/醫療/工業/金融/能源 BUY_NEW 323.66        $128,723       397 股     311.92     335.29         1.29%      良好
2317.TW           鴻海     台股電子/半導體/供應鏈 BUY_NEW 248.50        $128,408       516 股     239.23     264.83         1.28%      良好
2327.TW           國巨     台股電子/半導體/供應鏈 BUY_NEW 545.00        $128,280       235 股  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>